In [ ]:
## trying to reduce size of satellite data

# Start with one location to understand how the code works, then go from there.

# I previously downloaded monthly sst from satellite data, and need to reframe data aquisition
- need daily data at each buoy site
- want 4-12 km resolution 
- will stick to 2019-2020 time window to see what we can get

In [2]:
# ============================================================
# 7-DAY TEST: Download daily ~1km MUR SST from ERDDAP (site-scale)
# Copy/paste this entire cell into Jupyter and run.
# ============================================================

from __future__ import annotations

import math
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import requests
import xarray as xr


# -----------------------------
# ERDDAP / Dataset config
# -----------------------------
ERDDAP_SERVER = "https://spraydata.ucsd.edu/erddap"
DATASET_ID = "jplMURSST41"
VAR = "analysed_sst"  # Kelvin (we'll convert to Celsius)


# -----------------------------
# Helpers
# -----------------------------
@dataclass(frozen=True)
class BBox:
    lat_min: float
    lat_max: float
    lon_min: float
    lon_max: float


def bbox_km(lat: float, lon: float, box_km: float = 12.0) -> BBox:
    """Create a lat/lon bounding box (approx) with side length ~box_km centered at (lat, lon)."""
    half = box_km / 2.0
    dlat = half / 111.32
    dlon = half / (111.32 * math.cos(math.radians(lat)))
    return BBox(lat - dlat, lat + dlat, lon - dlon, lon + dlon)


def build_griddap_nc_url(
    server: str,
    dataset: str,
    var: str,
    start_iso: str,
    end_iso: str,
    box: BBox,
    stride: int = 1,
) -> str:
    """
    Build an ERDDAP griddap URL for a subset request.
    stride=1 = full resolution; stride=2 = downsample (faster).
    """
    server = server.rstrip("/")
    query = (
        f"{var}"
        f"[({start_iso}):{stride}:({end_iso})]"
        f"[({box.lat_min}):{stride}:({box.lat_max})]"
        f"[({box.lon_min}):{stride}:({box.lon_max})]"
    )
    return f"{server}/griddap/{dataset}.nc?{query}"


def download_with_retries(
    url: str,
    out_path: Path,
    *,
    timeout: tuple[int, int] = (15, 300),
    max_retries: int = 5,
) -> None:
    """
    Download a file with retry/backoff to handle slow ERDDAP responses.
    timeout=(connect_seconds, read_seconds).
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                with out_path.open("wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            return
        except Exception as e:
            last_err = e
            sleep_s = 2 ** (attempt - 1)
            print(f"Download failed (attempt {attempt}/{max_retries}): {type(e).__name__}: {e}")
            if attempt < max_retries:
                print(f"Retrying after {sleep_s}s...")
                time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {max_retries} attempts. Last error: {last_err}")


def mur_daily_site_mean_from_nc(nc_path):
    """
    Open MUR subset NetCDF and compute daily site-mean SST in Celsius.
    Handles unit differences (Kelvin vs Celsius) safely.
    """
    import xarray as xr
    import pandas as pd

    VAR = "analysed_sst"

    ds = xr.open_dataset(nc_path, decode_times=True, mask_and_scale=True)

    da = ds[VAR]
    units = (da.attrs.get("units") or "").strip().lower()

    # Convert only if units are Kelvin
    if units in {"k", "kelvin", "degrees_k", "degree_k"}:
        sst_c = da - 273.15
    else:
        sst_c = da

    ts = sst_c.mean(dim=("latitude", "longitude"), skipna=True)\
              .to_dataframe(name="sat_sst_c")\
              .reset_index()

    ts["date"] = pd.to_datetime(ts["time"]).dt.floor("D")

    daily = (
        ts.groupby("date", as_index=False)
          .agg(sat_sst_c=("sat_sst_c", "mean"))
          .sort_values("date")
    )

    return daily


# -----------------------------
# 7-day test parameters
# -----------------------------
SITE_NAME = "La_Push"
CENTER_LAT = 47.97
CENTER_LON = -124.95

BOX_KM_SIZE = 12.0   # site-scale (try 8.0 or 12.0)
STRIDE = 1           # set to 2 if you want faster
START_DATE = "2019-01-01"
END_DATE = "2019-01-07"

# MUR often uses ~09:00Z timestamps; we request with 09:00Z for consistency.
START_ISO = f"{START_DATE}T09:00:00Z"
END_ISO = f"{END_DATE}T09:00:00Z"

# Cache output
CACHE_DIR = Path("data/mur_cache") / SITE_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NC_PATH = CACHE_DIR / f"mur_{START_DATE}_{END_DATE}_stride{STRIDE}_box{int(BOX_KM_SIZE)}km.nc"

# -----------------------------
# Build URL, download, compute daily mean
# -----------------------------
box = bbox_km(CENTER_LAT, CENTER_LON, box_km=BOX_KM_SIZE)
url = build_griddap_nc_url(ERDDAP_SERVER, DATASET_ID, VAR, START_ISO, END_ISO, box, stride=STRIDE)

print("Request URL:\n", url)
print("Downloading to:\n", NC_PATH)

if not NC_PATH.exists():
    download_with_retries(url, NC_PATH, timeout=(15, 300), max_retries=5)
else:
    print("Using cached file (already downloaded).")

daily_sst = mur_daily_site_mean_from_nc(NC_PATH)
daily_sst

Request URL:
 https://spraydata.ucsd.edu/erddap/griddap/jplMURSST41.nc?analysed_sst[(2019-01-01T09:00:00Z):1:(2019-01-07T09:00:00Z)][(47.916101329500535):1:(48.02389867049946)][(-125.03050349614874):1:(-124.86949650385127)]
 data\mur_cache\La_Push\mur_2019-01-01_2019-01-07_stride1_box12km.nc
Using cached file (already downloaded).


,date,sat_sst_c
0,2019-01-01,9.712722
1,2019-01-02,9.461519
2,2019-01-03,9.994358
3,2019-01-04,9.916396
4,2019-01-05,9.987690
5,2019-01-06,10.288545
6,2019-01-07,10.371299


In [3]:
from pathlib import Path

def run_7day_test(site_name, lat, lon, start="2019-01-01", end="2019-01-07", box_km=12.0, stride=1):
    """
    Quick validation that MUR daily SST downloads + site-mean works for a buoy.
    """
    global NC_PATH  # reuse variable name from your prior cell style (optional)

    cache_dir = Path("data/mur_cache") / site_name
    cache_dir.mkdir(parents=True, exist_ok=True)

    start_iso = f"{start}T09:00:00Z"
    end_iso = f"{end}T09:00:00Z"

    box = bbox_km(lat, lon, box_km=box_km)
    url = build_griddap_nc_url(ERDDAP_SERVER, DATASET_ID, VAR, start_iso, end_iso, box, stride=stride)

    NC_PATH = cache_dir / f"mur_{start}_{end}_stride{stride}_box{int(box_km)}km.nc"
    print(f"\n=== {site_name} ===")
    print("URL:", url)
    print("NC :", NC_PATH)

    if not NC_PATH.exists():
        download_with_retries(url, NC_PATH, timeout=(15, 300), max_retries=5)
    else:
        print("Using cached file.")

    daily = mur_daily_site_mean_from_nc(NC_PATH)
    return daily

In [4]:
daily_cce2 = run_7day_test("CCE2_SoCal", 34.324, -120.814)
daily_cce2


=== CCE2_SoCal ===
URL: https://spraydata.ucsd.edu/erddap/griddap/jplMURSST41.nc?analysed_sst[(2019-01-01T09:00:00Z):1:(2019-01-07T09:00:00Z)][(34.270101329500534):1:(34.37789867049946)][(-120.87926351766104):1:(-120.74873648233894)]
NC : data\mur_cache\CCE2_SoCal\mur_2019-01-01_2019-01-07_stride1_box12km.nc


,date,sat_sst_c
0,2019-01-01,13.349089
1,2019-01-02,13.630917
2,2019-01-03,13.828881
3,2019-01-04,13.789458
4,2019-01-05,13.839732
5,2019-01-06,13.649405
6,2019-01-07,13.687244


## Now that I know I can get satellite data, I need to narrow down what I am going to get:
- what years have the best buoy data?
- after I know this, I can choose what satellite data to download

In [8]:
import pandas as pd

# EDIT PATH to your buoy cleaned file
BUOY_CSV = r"C:/Users/Owner/OneDrive - UW/Desktop/Grad School/coursework/machine learning/Geoceanographers/TeamProject/GeOceanProject/data/processed/buoy_data_cleaned.csv"

buoy = pd.read_csv(BUOY_CSV)

# Adjust these column names if yours differ
TIME_COL = "datetime"
LOC_COL = "location"
PCO2_COL = "pco2_sw_sat"

buoy[TIME_COL] = pd.to_datetime(buoy[TIME_COL], errors="coerce", utc=True).dt.tz_convert(None)
buoy[PCO2_COL] = pd.to_numeric(buoy[PCO2_COL], errors="coerce")

buoy = buoy.dropna(subset=[TIME_COL, LOC_COL, PCO2_COL])
buoy["year"] = buoy[TIME_COL].dt.year

counts = (
    buoy.groupby([LOC_COL, "year"], as_index=False)
        .agg(pco2_n=(PCO2_COL, "count"))
        .sort_values([LOC_COL, "year"])
)

# Pivot table: locations as rows, years as columns
pivot = counts.pivot(index=LOC_COL, columns="year", values="pco2_n").fillna(0).astype(int)

# Best year by total records across all locations
year_totals = pivot.sum(axis=0).sort_values(ascending=False)

print("Total pCO2 records by year (all locations):")
display(year_totals)

print("\nPer-location pCO2 records by year:")
display(pivot)

best_year = int(year_totals.index[0])
print("\nSuggested best year:", best_year)

Total pCO2 records by year (all locations):


year
2022    9127
2019    9002
2021    8717
2023    7734
2018    6616
2017    5689
2020    5434
2015    4726
2016    3913
2024    2801
2025    2515
2014    1161
2013     669
dtype: int64


Per-location pCO2 records by year:


year,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
location,,,,,,,,,,,,,
First Landing,0,0,0,0,0,2231,1685,1696,0,0,0,0,0
Grays Reef Georgia,0,0,0,1157,2098,321,1830,0,0,2557,1332,0,0
LA Buoy,0,0,0,0,0,0,1684,562,2338,2536,1264,0,0
La Push,0,0,0,0,0,600,755,0,1521,407,0,0,0
SE Bering Sea,0,0,0,0,912,1001,150,0,1935,925,1899,550,987
South Pacific,669,1161,2739,708,1059,0,0,832,0,595,1579,528,973
Southern California,0,0,1987,2048,1620,2463,2898,2344,2923,2107,1660,1723,555



Suggested best year: 2022


## after looking at buoy data, I decided to try and work with 2022 since it has the best overall data available, but does not cover all buoys. We can teak this later if needed. 

In [9]:
# ============================================================
# DOWNLOAD DAILY ~1km MUR SST FOR 2022 IN MONTHLY CHUNKS (12km)
# ============================================================

from __future__ import annotations

import math
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import requests
import xarray as xr


# -----------------------------
# CONFIG
# -----------------------------
YEAR = 2022
START_DATE = f"{YEAR}-01-01"
END_DATE = f"{YEAR}-12-31"

BOX_KM = 12.0
STRIDE = 1  # full resolution; set to 2 if you need faster

ERDDAP_SERVER = "https://spraydata.ucsd.edu/erddap"
DATASET_ID = "jplMURSST41"
VAR = "analysed_sst"  # unit may be C or K depending on server output


# -----------------------------
# BUOYS (lat, lon)
# -----------------------------
BUOYS = {
    "Grays_Reef": (31.40, -80.87),
    "LA_Buoy": (34.324, -120.814),
    "SE_Bering_Sea": (58.87, -164.06),
    "Southern_California": (33.55, -118.4),
    "South_Pacific": (0.0, -155.0),
}


# -----------------------------
# HELPERS
# -----------------------------
@dataclass(frozen=True)
class BBox:
    lat_min: float
    lat_max: float
    lon_min: float
    lon_max: float


def bbox_km(lat: float, lon: float, box_km: float) -> BBox:
    """Approx km box -> lat/lon bounds."""
    half = box_km / 2.0
    dlat = half / 111.32
    dlon = half / (111.32 * math.cos(math.radians(lat)))
    return BBox(lat - dlat, lat + dlat, lon - dlon, lon + dlon)


def build_griddap_nc_url(server: str, dataset: str, var: str, start_iso: str, end_iso: str, box: BBox, stride: int) -> str:
    """ERDDAP griddap subset URL."""
    server = server.rstrip("/")
    query = (
        f"{var}"
        f"[({start_iso}):{stride}:({end_iso})]"
        f"[({box.lat_min}):{stride}:({box.lat_max})]"
        f"[({box.lon_min}):{stride}:({box.lon_max})]"
    )
    return f"{server}/griddap/{dataset}.nc?{query}"


def download_with_retries(url: str, out_path: Path, *, timeout=(15, 300), max_retries: int = 5) -> None:
    """Download with exponential backoff for ERDDAP slowness."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                with out_path.open("wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            return
        except Exception as e:
            last_err = e
            wait_s = 2 ** (attempt - 1)
            print(f"Download failed (attempt {attempt}/{max_retries}): {type(e).__name__}: {e}")
            if attempt < max_retries:
                print(f"Retrying after {wait_s}s...")
                time.sleep(wait_s)

    raise RuntimeError(f"Failed after {max_retries} attempts. Last error: {last_err}")


def monthly_ranges(start: str, end: str) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """
    Create month-by-month inclusive date ranges from start..end.
    Example: 2022-01-01..2022-01-31, 2022-02-01..2022-02-28, ...
    """
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)

    month_starts = pd.date_range(start=start_ts, end=end_ts, freq="MS")
    ranges: list[tuple[pd.Timestamp, pd.Timestamp]] = []
    for ms in month_starts:
        me = (ms + pd.offsets.MonthEnd(0))
        me = min(me, end_ts)
        ranges.append((ms, me))
    return ranges


def site_daily_mean_from_nc(nc_path: Path) -> pd.DataFrame:
    """Open subset NetCDF and compute daily site-scale mean SST in Celsius."""
    ds = xr.open_dataset(nc_path, decode_times=True, mask_and_scale=True)
    da = ds[VAR]
    units = (da.attrs.get("units") or "").strip().lower()

    # Only convert if clearly Kelvin
    if units in {"k", "kelvin", "degrees_k", "degree_k"}:
        da = da - 273.15

    ts = da.mean(dim=("latitude", "longitude"), skipna=True).to_dataframe(name="sat_sst_c").reset_index()
    ts["date"] = pd.to_datetime(ts["time"]).dt.floor("D")
    daily = ts.groupby("date", as_index=False).agg(sat_sst_c=("sat_sst_c", "mean")).sort_values("date")
    return daily


# -----------------------------
# MAIN: download monthly chunks
# -----------------------------
CACHE_ROOT = Path("data/mur_cache_2022_monthly")
OUT_ROOT = Path("data")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

ranges = monthly_ranges(START_DATE, END_DATE)
all_sites = []

for site, (lat, lon) in BUOYS.items():
    print(f"\n==== {site} ====")
    box = bbox_km(lat, lon, BOX_KM)
    site_cache = CACHE_ROOT / site
    site_cache.mkdir(parents=True, exist_ok=True)

    parts = []
    for (s, e) in ranges:
        # MUR often reports daily values at ~09:00Z
        s_iso = s.strftime("%Y-%m-%dT09:00:00Z")
        e_iso = e.strftime("%Y-%m-%dT09:00:00Z")

        nc_path = site_cache / f"mur_{s.date()}_{e.date()}_stride{STRIDE}_box{int(BOX_KM)}km.nc"
        if not nc_path.exists():
            url = build_griddap_nc_url(ERDDAP_SERVER, DATASET_ID, VAR, s_iso, e_iso, box, STRIDE)
            print(f"Downloading {s.date()}..{e.date()}")
            download_with_retries(url, nc_path)
        else:
            print(f"Cached {s.date()}..{e.date()}")

        parts.append(site_daily_mean_from_nc(nc_path))

    site_df = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["date"]).sort_values("date")
    site_df["location"] = site
    site_df["center_lat"] = lat
    site_df["center_lon"] = lon
    site_df["box_km"] = BOX_KM

    out_site = OUT_ROOT / f"mur_daily_{YEAR}_{site}.csv"
    site_df.to_csv(out_site, index=False)
    print(f"Wrote {out_site} rows={len(site_df)}")

    all_sites.append(site_df)

combined = pd.concat(all_sites, ignore_index=True)
out_all = OUT_ROOT / f"mur_daily_{YEAR}_ALL_SITES.csv"
combined.to_csv(out_all, index=False)
print(f"\nWrote {out_all} rows={len(combined)}")


==== Grays_Reef ====
Wrote data\mur_daily_2022_Grays_Reef.csv rows=365

==== LA_Buoy ====
Wrote data\mur_daily_2022_LA_Buoy.csv rows=365

==== SE_Bering_Sea ====
Wrote data\mur_daily_2022_SE_Bering_Sea.csv rows=365

==== Southern_California ====
Wrote data\mur_daily_2022_Southern_California.csv rows=365

==== South_Pacific ====
Wrote data\mur_daily_2022_South_Pacific.csv rows=365

Wrote data\mur_daily_2022_ALL_SITES.csv rows=1825


In [10]:
from pathlib import Path

p = (Path.cwd() / "data" / "mur_daily_2022_ALL_SITES.csv").resolve()
print("Full path:", p)
print("Exists:", p.exists())

Full path: C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\Geoceanographers\TeamProject\GeOceanProject\notebooks\02_data_preparation\data\mur_daily_2022_ALL_SITES.csv
Exists: True


In [11]:
from pathlib import Path
import shutil

# 1) Set your project root (GeOceanProject folder)
PROJECT_ROOT = Path(r"C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\Geoceanographers\TeamProject\GeOceanProject")

# 2) Source paths (where the last script wrote them)
SRC_DATA_DIR = (Path.cwd() / "data").resolve()
SRC_CSV_ALL = SRC_DATA_DIR / "mur_daily_2022_ALL_SITES.csv"
SRC_CACHE_DIR = SRC_DATA_DIR / "mur_cache_2022_monthly"

# 3) Destination paths (your desired structure)
DST_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "mur"
DST_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "mur" / "2022"

DST_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DST_RAW_DIR.mkdir(parents=True, exist_ok=True)

print("SRC_DATA_DIR:", SRC_DATA_DIR)
print("Found ALL_SITES CSV:", SRC_CSV_ALL.exists())
print("Found cache dir:", SRC_CACHE_DIR.exists())
print("DST_PROCESSED_DIR:", DST_PROCESSED_DIR)
print("DST_RAW_DIR:", DST_RAW_DIR)

# 4) Move the combined CSV
if SRC_CSV_ALL.exists():
    shutil.move(str(SRC_CSV_ALL), str(DST_PROCESSED_DIR / SRC_CSV_ALL.name))

# 5) Move any per-site CSVs
for p in SRC_DATA_DIR.glob("mur_daily_2022_*.csv"):
    if p.name != "mur_daily_2022_ALL_SITES.csv":
        shutil.move(str(p), str(DST_PROCESSED_DIR / p.name))

# 6) Move the cache directory (raw .nc files)
if SRC_CACHE_DIR.exists():
    dst_cache = DST_RAW_DIR / SRC_CACHE_DIR.name  # keep folder name
    # If destination exists, merge copy + delete source (safe)
    if dst_cache.exists():
        shutil.copytree(SRC_CACHE_DIR, dst_cache, dirs_exist_ok=True)
        shutil.rmtree(SRC_CACHE_DIR)
    else:
        shutil.move(str(SRC_CACHE_DIR), str(dst_cache))

print("Done moving.")

SRC_DATA_DIR: C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\Geoceanographers\TeamProject\GeOceanProject\notebooks\02_data_preparation\data
Found ALL_SITES CSV: True
Found cache dir: True
DST_PROCESSED_DIR: C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\Geoceanographers\TeamProject\GeOceanProject\data\processed\mur
DST_RAW_DIR: C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\Geoceanographers\TeamProject\GeOceanProject\data\raw\mur\2022
Done moving.
